In [ ]:
# Cell 1: Install packages (OFFLINE MODE)

# Replace this with your copied path!
wheels_path = "/kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels" 

# Install everything strictly from your local folder
!pip install -U --no-index --find-links $wheels_path transformers peft bitsandbytes trl accelerate datasets

print("✅ Offline installation complete!")

In [ ]:
import os
import ast
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

# Fast matrix multiplication
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
# We MUST use the exact same logic from your training notebook!
def parse_and_join(text):
    try:
        clean_text = text.replace('\\/', '/')
        parsed_list = ast.literal_eval(clean_text)
        return "\n\n".join(parsed_list)
    except Exception:
        return str(text)

def smart_truncate(text, max_words):
    words = str(text).split()
    if len(words) <= max_words:
        return text
    head_len = int(max_words * 0.30)
    tail_len = max_words - head_len
    return " ".join(words[:head_len]) + "\n\n... [TRUNCATED] ...\n\n" + " ".join(words[-tail_len:])

def format_test_prompt(row):
    prompt = parse_and_join(row['prompt'])
    res_a = parse_and_join(row['response_a'])
    res_b = parse_and_join(row['response_b'])
    
    prompt_trunc = smart_truncate(prompt, max_words=450)
    res_a_trunc = smart_truncate(res_a, max_words=525)
    res_b_trunc = smart_truncate(res_b, max_words=525)
    
    template = f"""### Instruction:
Given a prompt and two responses, determine which response is better.

### Prompt:
{prompt_trunc}

### Response A:
{res_a_trunc}

### Response B:
{res_b_trunc}

### Answer:
"""
    return template.encode('utf-8', errors='replace').decode('utf-8')

In [ ]:
# 1. Paths (Update adapter_path to match your newly created dataset!)
base_model_id = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1" 
adapter_path = "/kaggle/input/notebooks/azizkarray/llm-classification-finetuning-comp/qwen-chatbot-arena-final"

print("🤖 Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)

print("🧠 Loading Base Model in pure FP16 (No BitsAndBytes!)...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=3,
    device_map="auto",
    torch_dtype=torch.float16, # <-- Pure 16-bit, no on-the-fly dequantization!
    attn_implementation="sdpa", # <-- PyTorch's optimized attention
    local_files_only=True
)
base_model.config.pad_token_id = tokenizer.pad_token_id

print("🧩 Attaching Trained LoRA Adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=False)
model.eval()

# Optional but recommended: Merge adapters into the base model to remove routing overhead
print("🔗 Merging adapters into base weights for maximum speed...")
model = model.merge_and_unload() 

print("✅ Model is fully loaded and ready for hyper-speed inference!")

In [ ]:
print("📂 Loading hidden test data...")
test_df = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")

print("⚙️ Formatting prompts to match training...")
test_df['formatted_prompt'] = test_df.apply(format_test_prompt, axis=1)

all_probs = []
# CRANK THIS UP! Try 16, 32, or 64. If it crashes, just restart the session and lower it.
batch_size = 64  

print(f"🚀 Generating predictions for {len(test_df)} rows at batch_size {batch_size}...")
for i in range(0, len(test_df), batch_size):
    batch_prompts = test_df['formatted_prompt'].iloc[i : i + batch_size].tolist()
    
    inputs = tokenizer(
        batch_prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    ).to("cuda")
    
    with torch.no_grad():
        logits = model(**inputs).logits
        
        # Because we loaded in float16, we still want to cast to float32 for safety with NumPy
        probs = torch.softmax(logits, dim=-1).float().cpu().numpy()
        all_probs.extend(probs)

print("📝 Creating submission.csv...")
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': [p[0] for p in all_probs],
    'winner_model_b': [p[1] for p in all_probs],
    'winner_tie': [p[2] for p in all_probs]
})

submission.to_csv("submission.csv", index=False)
print("🏆 submission.csv created successfully! Ready for scoring!")